![HydroCycle](images/hydro_5cycle.jpg)

# Retrieve and Analyze Hydrology data for a watershed of interest

To make predictions for reservoir operations, water supply, flood control, etc, we need to collect data to train/calibrate hydrologic models. This includes streamflow, current environmental conditions (e.g., snow water equivalent), and future weather predictions. This exercise will build on the previous SNOTEL module to work towards building a hydrologic module.

Need to find a station? Use the [USGS NWIS mapper system](https://apps.usgs.gov/nwismapper/)


Click the link and explore!

# 1. Delineated Watershed Map Upstream of a NWIS Site
The following code uses the pynhd and folium packages to create an interactive map of a watershed from a USGS gauge ID.

In our exercise, we are tasked with identifying all SNOTEL sites upstream of Hetch Hetchy Reservoir on the Tuolumne River. The user can search for "USGS streamflow Tuolumne River" and serveral locations will pop up. Site [11274790](https://waterdata.usgs.gov/monitoring-location/11274790/#dataTypeId=continuous-00065-0&period=P7D&showMedian=false) is the site of interest for this assessment 

In [ ]:
from pynhd import NLDI, WaterData, NHDPlusHR, GeoConnex
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import warnings
import importlib
warnings.filterwarnings("ignore")


Define the watershed outlet using NWIS site id. Create a map object that we'll add layers to.

In [ ]:
nldi = NLDI()
usgs_gage_id = "11274790" # NWIS id for Tuolumne river at the mouth of Hetch Hetchy Reservoir


## Get meterological information data for our area of interest using GEE and NLDAS2

We will be using the GEE python api to retrived NLDAS2 meteorological data from the [NLDAS Earth Engine Catalog](https://developers.google.com/earth-engine/datasets/catalog/NASA_NLDAS_FORA0125_H002).

![GEE_NLDAS](images/NASA_NLDAS_FORA0125_H002_sample.png)


In [ ]:
# Get geometry and ensure CRS is correct
basin = nldi.get_basins(usgs_gage_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
basin_polygon_coords = list(geometry.exterior.coords)

In [ ]:
#we can get hourly data  but it can take a long time and crash if the start and end date are too far apart
hourly_NLDAS_df = getData.get_NLDAS_hourly(basin_polygon_coords)
hourly_NLDAS_df.head()



In [ ]:
import sys
sys.path.append('/home/civil/Data-Acquisition-Processing-Analysis/supporting_scripts')
import getData
import importlib
importlib.reload(getData)

#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df1 = getData.get_NLDAS_daily(basin_polygon_coords, begin_date='2006-01-01', end_date='2012-01-1')
Daily_NLDAS_df1.head()

In [ ]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df_2 = getData.get_NLDAS_daily(basin_polygon_coords, begin_date='2012-01-1', end_date='2018-01-1')
Daily_NLDAS_df_2.head()

In [ ]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df_3 = getData.get_NLDAS_daily(basin_polygon_coords, begin_date='2018-01-1', end_date='2022-01-1')
Daily_NLDAS_df_3.head()

In [ ]:
#combine the three dataframes into one
Daily_NLDAS_df = pd.concat([Daily_NLDAS_df1, Daily_NLDAS_df_2, Daily_NLDAS_df_3], ignore_index=False)
#set 
Daily_NLDAS_df.head()

In [ ]:
Daily_NLDAS_df.tail()

## Data exploration

### Lets examine basin longwave and shortwave radiation

In [ ]:
#For year 2019, plot all SWE_cm columns
met_df_2019 = Daily_NLDAS_df.loc['2019-10-01':'2020-09-30'].copy()
#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Radiation (W/m²)', color='black', fontsize=12, fontweight='bold')
ax1.plot(met_df_2019.index, met_df_2019['longwave_radiation'], linewidth=2, label = 'LW Radiation', color='darkred')
ax1.plot(met_df_2019.index, met_df_2019['shortwave_radiation'], linewidth=2, label = 'SW Radiation', color='darkblue')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, alpha=0.3)

#show a legend
ax1.legend()

# Title and Layout
plt.title(f" Radiation for water year 2019 in the Basin \n Upstream of USGS gage: {usgs_gage_id}", fontsize=14, pad=15)
fig.tight_layout()
plt.show()

### Lets look at temperature

In [ ]:
met_df_2019.columns

In [ ]:
#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Temperature (°C)', color='darkorange', fontsize=12, fontweight='bold')
ax1.plot(met_df_2019.index, met_df_2019['temperature'], linewidth=2, label = 'Daily Mean Temperature', color='red')
ax1.tick_params(axis='y', labelcolor='darkorange')
ax1.grid(True, alpha=0.3)

#show a legend
ax1.legend()

# Title and Layout
plt.title(f"Temperature for water year 2019 in the Basin \n Upstream of USGS gage: {usgs_gage_id}", fontsize=14, pad=15)
fig.tight_layout()
plt.show()

### Precipitation and temperature

Use a dual y-axies to plot both on the same plot. 

What insights can we make about the type of precipitation?

In [ ]:
def align_yaxis(ax1, ax2):
    """Align zeros of the two axes by adjusting their limits."""
    # Get current limits
    y1_min, y1_max = ax1.get_ylim()
    y2_min, y2_max = ax2.get_ylim()

    # Calculate the ratio of the zero point relative to the total range
    # Example: If zero is exactly in the middle, ratio = 0.5
    y1_ratio = -y1_min / (y1_max - y1_min)
    y2_ratio = -y2_min / (y2_max - y2_min)

    # Adjust the axis with the smaller ratio to match the larger one
    if y1_ratio < y2_ratio:
        ax1.set_ylim(bottom=-y1_max * y2_ratio / (1 - y2_ratio))
    else:
        ax2.set_ylim(bottom=-y2_max * y1_ratio / (1 - y1_ratio))

In [ ]:
#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Temperature (°C)', color='orange', fontsize=12, fontweight='bold')
ax1.plot(met_df_2019.index, met_df_2019['temperature'], linewidth=2, label = 'Daily Mean Temperature', color='orange')
#ax1.plot(met_df_2019.index, met_df_2019['prcp_mm_day'], linewidth=2, label = 'Daily Precipitation', color='blue')
ax1.tick_params(axis='y', labelcolor='orange')
#make secondary y-axis for precipitation
ax2 = ax1.twinx()
ax2.plot(met_df_2019.index, met_df_2019['total_precipitation'], linewidth=2, label = 'Daily Precipitation', color='blue')
ax2.set_ylabel('Precipitation (mm)', color='blue', fontsize=12)
ax1.grid(True, alpha=0.3)
align_yaxis(ax1, ax2)

#Ask both axes for their handles and labels
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

#Combine them and create one legend
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')


# Title and Layout
plt.title(f"Precipitation and Temperature for water year 2019 in the Basin \n Upstream of USGS gage: {usgs_gage_id}", fontsize=14, pad=15)
fig.tight_layout()
plt.show()

### Plot wind

wind_u (Zonal Wind): Represents the east-west component. A positive value indicates wind blowing toward the east (from the west).

wind_v (Meridional Wind): Represents the north-south component. A positive value indicates wind blowing toward the north (from the south)

In [ ]:
met_df_2019.columns


In [ ]:
#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Wind Speed (m/s)', color='black', fontsize=12, fontweight='bold')
ax1.plot(met_df_2019.index, met_df_2019['wind_u'], linewidth=2, label = 'Zonal Wind Speed', color='blue')
ax1.plot(met_df_2019.index, met_df_2019['wind_v'], linewidth=2, label = 'Meridional Wind Speed', color='green')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, alpha=0.3)

#show a legend
ax1.legend()

# Title and Layout
plt.title(f"Wind Speed for water year 2019 in the Basin \n Upstream of USGS gage: {usgs_gage_id}", fontsize=14, pad=15)
fig.tight_layout()
plt.show()

### save the data for future use!

In [ ]:
#save the cleaned dataframe to a csv file
# Use the getData module to retrieve data 
OutputFolder = 'files/NLDAS'
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)
Daily_NLDAS_df.to_csv(f'{OutputFolder}/NLDAS_{usgs_gage_id}.csv')

## Exercise: Fetch a variable of your choice from Earth Engine

In this exercise, you are tasked with getting another variable from the [Google Earth Engine data catelog](https://developers.google.com/earth-engine/datasets/), something like the [MODIS Snow Covered Daily product](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD10A1). Add your function to your getData.py script and create a data processing function to bring to a daily resolution (if necessary) into dataprocessing.py. Plot you data either as a time series (as we have previously done) or over the basin for a single day.

In [ ]:
import sys
sys.path.append('/home/civil/Data-Acquisition-Processing-Analysis/supporting_scripts')
import getData, dataprocessing
import importlib
importlib.reload(getData)
importlib.reload(dataprocessing)
import matplotlib.pyplot as plt

# Fetch GRIDMET precipitation for WY2025
gridmet_df = getData.get_GRIDMET_daily(
    basin_polygon_coords,
    begin_date='2024-10-01',
    end_date='2025-04-01',
    variable='pr'
)

# Process
gridmet_df = dataprocessing.processGRIDMET(gridmet_df, variable='pr')

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(gridmet_df.index, gridmet_df['GRIDMET_pr'], color='steelblue', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Precipitation (mm)')
ax.set_title('GRIDMET Daily Precipitation - Bear River Basin WY2025')
plt.tight_layout()
plt.show()